# HW1: Data Preprocessing, KPI, and PCA

## Introduction: Exploring the Drivers of Hotel Booking Cancellations

Imagine you are a **data science consultant** who has just been hired by two hotels.  
The managers are worried about the growing number of **cancellations**, and they want to understand what drives this behavior so they can make better business decisions.  

Your job in this homework is to help the hotel managers uncover the **main reasons behind cancellations**.  
You will approach this problem using the **same structured workflow** you practiced in the Lab (remember when you were part of the airline’s *Customer Experience & Insights* team 😄).  

This time, instead of analyzing passenger satisfaction, you are analyzing hotel bookings. The process is the same:  

- Load and clean the dataset,  
- Explore it systematically,  
- Compute KPIs and summaries,  
- Apply PCA to simplify complexity,  
- And extract insights.  

At the end, you should be able to **report the key factors behind cancellations**.  
While this is not a formal requirement in the submission, try to **write a short text as if you were delivering recommendations to your client**. The goal is to practice turning analysis into **clear, data-driven advice** that managers can act on.  


**This homework is your first hands-on warm-up. By the end, you will be able to:**  
- Load a real dataset and perform a **light, deterministic clean**.  
- Compute **basic KPIs** and **categorical summaries** without plotting.  
- Build a **PCA-ready numeric matrix** (impute + standardize) and run **PCA**.  
- Interpret **explained variance ratios** and **dominant features** for principal components.  
- Submit clean, testable code that passes **VPL** auto-grading.


## Dataset & Columns: Quick Guide

**Data Source & Coverage**  
This homework uses the public **Hotel booking demand** dataset. It contains **booking-level records** from two hotels in Portugal - a **City Hotel** and a **Resort Hotel** - covering bookings around **2015–2017**.  
A copy is available on Kaggle: [Hotel booking demand (Mostipak)](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand).

> First time using Kaggle?
>Use the link above to read about the dataset, explore the documentation, and get familiar with the platform.


- **File:** `hotel_bookings.csv`  
- **Size:** **119390 rows × 32 columns**  
- **Goal of this HW:** understand **cancellations** and prepare numeric features for **PCA**.

**What’s inside:** 
This data set contains booking information for a city hotel and a resort hotel, and includes information such as when the booking was made, length of stay, the number of adults, children, and/or babies, and the number of available parking spaces, among other things. All personally identifying information has been removed from the data.

- Booking details (e.g., **lead time**, **stay length**, **ADR** price).  
- Channel/segment info (e.g., **market_segment**, **distribution_channel**, **deposit_type**).  
- Guest counts (**adults**, **children**, **babies**) and a flag for **repeated guests**.  
- Dates (arrival year/month/day, and **reservation_status_date** for final status).  
- The dataset is anonymized.

**Target / Outcome**  
- **`is_canceled`** — 0 (not canceled) or 1 (canceled).

**Key categoricals used in EDA**
- **`hotel`**
- **`deposit_type`**
- **`market_segment`**
- **`distribution_channel`**
- **`customer_type`**

**Numeric features used for PCA (14)**
- **`lead_time`**
- **`stays_in_weekend_nights`**
- **`stays_in_week_nights`**
- **`adults`**
- **`children`**
- **`babies`**
- **`previous_cancellations`**
- **`previous_bookings_not_canceled`**
- **`booking_changes`**
- **`days_in_waiting_list`**
- **`adr`**
- **`required_car_parking_spaces`**
- **`total_of_special_requests`**
- **`total_guests`** *(we will create this: adults + children + babies)*

> Tip: Don’t rename columns. VPL expects the standard names in your functions.


## Before You Start  

We will follow a **similar structured workflow** as in the lab. Since homeworks are automatically graded with the **VPL system**, and VPL cannot evaluate plots, we will check **numeric KPIs** instead of graphs.  

➡️ **Recommendation:** Create your own plots locally while working through the notebook. This extra practice will help you explore the data more deeply and strengthen your intuition.  


## Task 1: Load & Light Clean 
---

**What you’ll build:** a function `load_and_clean(csv_path)` that returns a **cleaned** `pandas.DataFrame` ready for analysis.  
You will **not** plot anything here. Keep the cleaning **deterministic** so everyone gets the same result.

In short: These steps make sure dates are real dates, numbers are real numbers, bookings make sense (no zero guests), and errors are flagged as missing values instead of breaking the dataset.

### Required steps (follow in this order)
1) **Convert reservation date column into proper date format**  
   - Convert the column `reservation_status_date` from text into a proper date format so it can be used in time-based analysis.
   - If a value doesn’t look like a valid date, mark it as NaT (Not a Time), instead of crashing the process.

2) **Convert numeric columns** (convert strings to numbers; invalids → `NaN`)  
   - Columns to convert (if present):  
     `is_canceled, lead_time, stays_in_weekend_nights, stays_in_week_nights, adults, children, babies,`  
     `previous_cancellations, previous_bookings_not_canceled, booking_changes, days_in_waiting_list, adr,`  
     `required_car_parking_spaces, total_of_special_requests, arrival_date_year, arrival_date_day_of_month`  
   - Use `pd.to_numeric(col, errors="coerce")` in a simple loop.
   - If a value can’t be converted (e.g., text in a number column), replace it with `NaN`. 

3) **Create total guests**  
   - Create new column: `total_guests = adults + children + babies` (row‑wise sum).  
   - If any of these are `NaN`, the sum will be `NaN` — that’s fine for now.

4) **Drop impossible rows**  
   - Keep only rows where `total_guests > 0`. (Removes bookings with 0 guests.)

5) **Handle negative prices**  
   - If `adr < 0`, set it to `NaN` (do **not** drop those rows).
   - ADR represents the average revenue earned per occupied room per day.

In [1]:
# If you changed hw1.py, reload the module so Jupyter sees your edits
import numpy as np
import importlib
import hw1 as hw
importlib.reload(hw)
from __future__ import annotations
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [3]:
PCA_FEATURES: List[str] = [
    "lead_time",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "days_in_waiting_list",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
    "total_guests",
]

In [4]:
def load_and_clean(csv_path: str) -> pd.DataFrame:
    """
    Task 1 — Load & Light Clean (simple instructions)

    What you will do:
    - Turn the reservation status date into a real date type.
    - Turn obvious numeric columns into real numbers (bad values become NaN).
    - Create a new column total_guests = adults + children + babies.
    - Remove rows where total_guests == 0 (bookings must have guests).
    - If adr (average daily rate) is negative, set it to NaN (do not drop the row).

    Why this matters:
    These steps make sure dates are real dates, numbers are real numbers,
    bookings make sense (no zero guests), and errors are marked as missing values.

    Expected shape on the official dataset:
    - (119210, 33) after cleaning.

    Return:
    - DataFrame with all original columns PLUS 'total_guests'.
    - Do NOT drop duplicates. Do NOT fill other missing values here.
    """

    # TODO 1 — Read the CSV file into a DataFrame
    
    # TODO 2 — Convert reservation_status_date into a proper datetime (invalid -> NaT)
    
    # TODO 3 — Convert obvious numeric columns to numbers (invalid -> NaN)
    numeric_cols = [
        "is_canceled", "lead_time", "stays_in_weekend_nights", "stays_in_week_nights",
        "adults", "children", "babies",
        "previous_cancellations", "previous_bookings_not_canceled",
        "booking_changes", "days_in_waiting_list", "adr",
        "required_car_parking_spaces", "total_of_special_requests",
        "arrival_date_year", "arrival_date_day_of_month",
    ]
    
    # TODO 4 — Create total_guests = adults + children + babies (row-wise sum)
    
    # TODO 5 — Keep only rows where total_guests > 0
    
    # TODO 6 — Handle negative prices: if adr < 0, set it to NaN (do NOT drop the row)
    
    # TODO 7 — Return the cleaned DataFrame
   

In [10]:
print(dir(pd))

['ArrowDtype', 'BooleanDtype', 'Categorical', 'CategoricalDtype', 'CategoricalIndex', 'DataFrame', 'DateOffset', 'DatetimeIndex', 'DatetimeTZDtype', 'ExcelFile', 'ExcelWriter', 'Flags', 'Float32Dtype', 'Float64Dtype', 'Grouper', 'HDFStore', 'Index', 'IndexSlice', 'Int16Dtype', 'Int32Dtype', 'Int64Dtype', 'Int8Dtype', 'Interval', 'IntervalDtype', 'IntervalIndex', 'MultiIndex', 'NA', 'NaT', 'NamedAgg', 'Period', 'PeriodDtype', 'PeriodIndex', 'RangeIndex', 'Series', 'SparseDtype', 'StringDtype', 'Timedelta', 'TimedeltaIndex', 'Timestamp', 'UInt16Dtype', 'UInt32Dtype', 'UInt64Dtype', 'UInt8Dtype', '__all__', '__builtins__', '__cached__', '__doc__', '__docformat__', '__file__', '__git_version__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_built_with_meson', '_config', '_is_numpy_dev', '_libs', '_pandas_datetime_CAPI', '_pandas_parser_CAPI', '_testing', '_typing', '_version_meson', 'annotations', 'api', 'array', 'arrays', 'bdate_range', 'compat', 'conca

In [67]:
df = pd.read_csv("hotel_bookings.csv")
df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"],errors="coerce")
df["reservation_status_date"].isna().sum()

np.int64(0)

In [68]:
numeric_cols = [
        "is_canceled", "lead_time", "stays_in_weekend_nights", "stays_in_week_nights",
        "adults", "children", "babies",
        "previous_cancellations", "previous_bookings_not_canceled",
        "booking_changes", "days_in_waiting_list", "adr",
        "required_car_parking_spaces", "total_of_special_requests",
        "arrival_date_year", "arrival_date_day_of_month",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col],errors='coerce')
    print(col,": null values are ",df[col].isna().sum())


is_canceled : null values are  0
lead_time : null values are  0
stays_in_weekend_nights : null values are  0
stays_in_week_nights : null values are  0
adults : null values are  0
children : null values are  4
babies : null values are  0
previous_cancellations : null values are  0
previous_bookings_not_canceled : null values are  0
booking_changes : null values are  0
days_in_waiting_list : null values are  0
adr : null values are  0
required_car_parking_spaces : null values are  0
total_of_special_requests : null values are  0
arrival_date_year : null values are  0
arrival_date_day_of_month : null values are  0


In [85]:
df["children"].isna().sum()

np.int64(0)

In [75]:
df["adults"].isna().sum()

np.int64(0)

In [76]:
df["babies"].isna().sum()

np.int64(0)

In [81]:
df.loc[df["children"] == np.nan , 'children'] = 0

In [84]:
df["children"] = df["children"].fillna(0)

In [86]:
df['total_guests'] = df['adults']+df["children"] + df["babies"]

In [29]:
print(dir(df))

['T', '_AXIS_LEN', '_AXIS_ORDERS', '_AXIS_TO_AXIS_NUMBER', '_HANDLED_TYPES', '__abs__', '__add__', '__and__', '__annotations__', '__array__', '__array_priority__', '__array_ufunc__', '__arrow_c_stream__', '__bool__', '__class__', '__contains__', '__copy__', '__dataframe__', '__dataframe_consortium_standard__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__divmod__', '__doc__', '__eq__', '__finalize__', '__firstlineno__', '__floordiv__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__iand__', '__ifloordiv__', '__imod__', '__imul__', '__init__', '__init_subclass__', '__invert__', '__ior__', '__ipow__', '__isub__', '__iter__', '__itruediv__', '__ixor__', '__le__', '__len__', '__lt__', '__matmul__', '__mod__', '__module__', '__mul__', '__ne__', '__neg__', '__new__', '__nonzero__', '__or__', '__pandas_priority__', '__pos__', '__pow__', '__radd__', '__rand__', '__rdivmod__', '__reduce__'

In [87]:
df = df[df['total_guests'] > 0]

In [88]:
df.shape

(119210, 33)

In [89]:
df.loc[df['adr']<0,'adr'] = np.nan

In [91]:
# ⚠️ Do not change this part!  
# These assertions mirror the VPL auto-grading checks.  
# Use them locally to confirm that your function works properly.

assert df.shape == (119210, 33)
assert "total_guests" in df and (df["total_guests"] > 0).all()
assert (df["adr"] < 0).sum() == 0

## Task 2: Basic Numeric KPIs
---

**What you’ll build:** a function `numeric_kpis(df)` that returns a small **dictionary of key numbers** describing the cleaned dataset from **Task 1**.  
No plots, no printing required, just compute and return the values.

In short: You’ll get a quick “health check” of the dataset: how big it is, how often bookings are canceled, how prices look at the high end, and how long stays typically are.

### Required steps

1) **Use the cleaned DataFrame from Task 1**  
   - Input to this function is the **already cleaned** `df` from `load_and_clean(...)`.

2) **Compute these KPIs and return them in a dict:**  
   - **`rows`** → number of rows (after cleaning).  
       
   - **`cols`** → number of columns.  
     
   - **`cancel_rate`** → average of `is_canceled`. (NaNs are ignored automatically)

   - **`adr_p95`** → 95th percentile of `adr` (a robust “high price” marker).  
    
   - **`avg_stay_len`** → mean of **total nights** per booking, where  
     `total_nights = stays_in_week_nights + stays_in_weekend_nights`.  
    

In [ ]:
# Reload again to reset any changes
importlib.reload(hw)

In [ ]:
k = hw.numeric_kpis(df)
display(k)

In [ ]:
def numeric_kpis(df: pd.DataFrame) -> Dict[str, float]:
    """
    Task 2 — Basic Numeric KPIs (simple instructions)

    What you will do:
    - Take the cleaned DataFrame from Task 1.
    - Compute a few key numbers (no plotting/printing).
    - Return them in a small dictionary.

    KPIs to compute:
    - rows          -> number of rows in the cleaned data
    - cols          -> number of columns
    - cancel_rate   -> mean of is_canceled
    - adr_p95       -> 95th percentile of adr
    - avg_stay_len  -> mean of total nights
                       (stays_in_week_nights + stays_in_weekend_nights)
    """

    # TODO 1 — rows

    # TODO 2 — cols

    # TODO 3 — cancel_rate

    # TODO 4 — adr_p95

    # TODO 5 — avg_stay_len

    # TODO 6 — package & return
    

In [92]:
rows,cols = df.shape

In [93]:
cancel_rate = df['is_canceled'].mean()

In [94]:
adr_p95 = df['adr'].quantile(0.95)

In [95]:
avg_stay_len = (df['stays_in_week_nights']+df['stays_in_weekend_nights']).mean()

In [97]:
k = {
    "rows":rows,
    "cols": cols,
    "cancel_rate":cancel_rate,
    "adr_p95":adr_p95,
    "avg_stay_len":avg_stay_len
}

In [98]:
# ⚠️ Do not change this part!  
# These assertions mirror the VPL auto-grading checks.  
# Use them locally to confirm that your function works properly.

assert k["rows"] == 119210 and k["cols"] == 33
assert abs(k["cancel_rate"] - 0.370766) < 1e-3
assert abs(k["adr_p95"] - 193.5) < 1e-6
assert abs(k["avg_stay_len"] - 3.426248) < 1e-3

## Task 3: Categorical EDA (no plots)
---

**What you’ll build:** a function `categorical_cancel_stats(df)` that summarizes **cancellation rates by category** and identifies which **market segment** has the highest cancellation rate among sufficiently common categories.  
No plots — just compute the numbers and return them.

In short: You’ll measure how cancellations differ across **hotel type** and **deposit policy**, and find the highest-risk **market segment** (only considering segments with enough data).

### Required steps

1) **Use the cleaned DataFrame from Task 1**  
   - Input to this function is the **already cleaned** `df` from `load_and_clean(...)`.

2) **Cancellation by hotel type** → `hotel_rates`  
   - Group by `hotel` and compute the **mean of `is_canceled`**.  
   - Convert to a Python dictionary.  

3) **Cancellation by deposit type** → `deposit_rates`  
   - Group by `deposit_type` and compute the **mean of `is_canceled`**.  
   - Convert to a dictionary.  

4) **Top market segment (n ≥ 500)** → `top_segment_500`  
   - Count rows per `market_segment` and **keep only categories with at least 500 rows**.  
   - Filter `df` to those segments, group by `market_segment`, compute the **mean cancellation rate**, and sort descending.  
   - Take the **first** one and return it as a **tuple**: `(segment_name, rate)` where `rate` is a float.  
    
5) **Return format (exact keys):**  
   ```python
   {
     "hotel_rates": <dict>,        # e.g., {"City Hotel": 0.xxx, "Resort Hotel": 0.yyy}
     "deposit_rates": <dict>,      # e.g., {"Non Refund": 0.xxx, "No Deposit": 0.yyy, "Refundable": 0.zzz}
     "top_segment_500": (<name>, <rate_float>)
   }


In [ ]:
# Reload again to reset any changes
importlib.reload(hw)

In [ ]:
cat = hw.categorical_cancel_stats(df)
cat

In [ ]:
# ============================ TASK 3 ============================
def categorical_cancel_stats(df: pd.DataFrame) -> Dict[str, object]:
    """
    Task 3 — Categorical EDA (simple instructions)

    What you will do:
    - Use the cleaned DataFrame from Task 1.
    - Compute cancellation rates (mean of is_canceled) by:
        * hotel        -> hotel_rates (dict)
        * deposit_type -> deposit_rates (dict)
    - Find the market segment with the highest cancel rate among
      segments with at least 500 rows -> ('segment', rate_float).
    """

    # TODO 1 — hotel_rates

    # TODO 2 — deposit_rates

    # TODO 3 — top_segment_500
    
    # TODO 4 — return dict

In [102]:
df['deposit_type'].unique()

array(['No Deposit', 'Refundable', 'Non Refund'], dtype=object)

In [103]:
df['hotel'].unique()

array(['Resort Hotel', 'City Hotel'], dtype=object)

In [113]:
check = df.groupby('hotel')['is_canceled'].mean()
print(check.to_dict())
cat ={}
cat["hotel_rates"] =check.to_dict() 

{'City Hotel': 0.4178593534858457, 'Resort Hotel': 0.27767373336329815}


In [114]:
check = df.groupby('deposit_type')['is_canceled'].mean()
print(check.to_dict())
cat["deposit_type"] =check.to_dict() 

{'No Deposit': 0.28401987344559215, 'Non Refund': 0.9936244601357374, 'Refundable': 0.2222222222222222}


In [144]:
check = df.groupby('market_segment')['is_canceled'].count()
check = check[check > 500]
segment_filtered = check.index.to_list()
segment_filtered

['Complementary',
 'Corporate',
 'Direct',
 'Groups',
 'Offline TA/TO',
 'Online TA']

In [145]:
df['is_canceled'].unique()

array([0, 1])

In [189]:
# df_new = df[df['market_segment'].isin(segment_filtered)]
check = df.groupby('market_segment')['is_canceled'].mean()
a = check.sort_values(ascending=False)
# print(dir(a))
# print(dir(a)
# a.index.to_list()
result = []
for i in a.index.to_list():
    if i in segment_filtered:
        result.append(i)
cat["top_segment_500"] = result
result

['Groups',
 'Online TA',
 'Offline TA/TO',
 'Corporate',
 'Direct',
 'Complementary']

In [190]:
# ⚠️ Do not change this part!  
# These assertions mirror the VPL auto-grading checks.  
# Use them locally to confirm that your function works properly.

assert "City Hotel" in cat["hotel_rates"] and "Resort Hotel" in cat["hotel_rates"]
assert cat["top_segment_500"][0] == "Groups"

## Task 4: Build PCA-ready Matrix
---

**What you’ll build:** a function `build_pca_matrix(df)` that takes the **cleaned** DataFrame from Task 1 and returns a **NumPy array** of numeric features, ready for PCA.  
No plots, no printing — just return the matrix.

In short: You will **pick 14 numeric features**, **fill missing values with the median**, and **standardize** each feature to mean 0 and standard deviation 1.

### Features to include (exactly these 14)
- `lead_time`  
- `stays_in_weekend_nights`  
- `stays_in_week_nights`  
- `adults`  
- `children`  
- `babies`  
- `previous_cancellations`  
- `previous_bookings_not_canceled`  
- `booking_changes`  
- `days_in_waiting_list`  
- `adr`  
- `required_car_parking_spaces`  
- `total_of_special_requests`  
- `total_guests`  *(created in Task 1)*

> Do **not** include the target `is_canceled`. No categorical encoding is needed for this task.

### Required steps
1) **Select the columns** above, in that order.  
   - You may store this list in a constant like `PCA_FEATURES` to keep it consistent.

2) **Impute missing values with the median**  
   - Use `sklearn.impute.SimpleImputer(strategy="median")`.  
   - Fit on your selected columns and transform to get a **fully numeric, NaN-free** array.

3) **Standardize (z-score) each feature**  
   - Use `sklearn.preprocessing.StandardScaler()`.  
   - Fit on the imputed data, then transform so each column has mean ~0 and std ~1.

4) **Return a NumPy array** (not a DataFrame)  
   - Shape must be `(n_samples, 14)` and contain **no NaNs**.


In [ ]:
# ============================ TASK 4 ============================
def build_pca_matrix(df: pd.DataFrame) -> np.ndarray:
    """
    Task 4 — Build PCA-ready Matrix (simple instructions)

    What you will do:
    - Use the cleaned DataFrame from Task 1.
    - Select EXACTLY the 14 numeric features in PCA_FEATURES (in order).
    - Median-impute missing values, then standardize (z-score).
    - Return a NumPy array with shape (n_samples, 14), no NaNs.
    """

    # TODO 1 — select columns in order
    
    # TODO 2 — median imputation
    
    # TODO 3 — standardize
    
    # TODO 4 — return array
    

In [193]:
imp = SimpleImputer(strategy='median')

In [194]:
df_numerical = df[PCA_FEATURES]

In [195]:
imputer_array = imp.fit_transform(df_numerical)

In [200]:
scaler = StandardScaler()

In [202]:
X_scaled = scaler.fit_transform(imputer_array)
X_scaled

array([[ 2.22587854, -0.93160599, -1.31737788, ..., -0.25494722,
        -0.72080191,  0.04004813],
       [ 5.92178491, -0.93160599, -1.31737788, ..., -0.25494722,
        -0.72080191,  0.04004813],
       [-0.90862434, -0.93160599, -0.79025693, ..., -0.25494722,
        -0.72080191, -1.35101768],
       ...,
       [-0.65599276,  1.0782163 ,  1.31822686, ..., -0.25494722,
         4.32414494,  0.04004813],
       [ 0.04576161,  1.0782163 ,  1.31822686, ..., -0.25494722,
        -0.72080191,  0.04004813],
       [ 0.94400721,  1.0782163 ,  2.37246875, ..., -0.25494722,
         1.80167151,  0.04004813]], shape=(119210, 14))

In [ ]:
# Reload again to reset any changes
importlib.reload(hw)

In [203]:
X = X_scaled
X.shape, np.isnan(X).sum()

((119210, 14), np.int64(0))

In [204]:
# ⚠️ Do not change this part!  
# These assertions mirror the VPL auto-grading checks.  
# Use them locally to confirm that your function works properly.

assert X.shape == (len(df), 14) and not np.isnan(X).any()

## Task 5: Run PCA and Inspect Results
---

**What you’ll build:** a function `run_pca(X, n_components=3)` that fits **Principal Component Analysis** on the PCA-ready matrix from **Task 4** and returns two things:  
1) the **explained variance ratios** for each component, and  
2) the **component loadings** (the weights for each original feature).

No plots are required — just compute and return the values.

In short: You’ll compress the 14 standardized features into 3 principal components, then report how much variance each component explains and what linear combinations (loadings) define them.

### Required steps

1) **Use the matrix from Task 4**  
   - Input is the NumPy array `X` produced by `build_pca_matrix(df)` (shape `(n_samples, 14)`, no NaNs).

2) **Initialize PCA**  
   - `from sklearn.decomposition import PCA`  
   - `pca = PCA(n_components=3)`

3) **Fit PCA on `X`**  
   - `pca.fit(X)`

4) **Return these two attributes**  
   - `pca.explained_variance_ratio_`  → array of length **3**  
   - `pca.components_`                → matrix of shape **(3, 14)**

> Do **not** return the PCA object itself. Return the **tuple** `(explained_variance_ratio_, components_)`.


In [205]:
def run_pca(X: np.ndarray, n_components: int = 3) -> Tuple[np.ndarray, np.ndarray]:
    """
    Task 5 — Run PCA (simple instructions)

    What you will do:
    - Take the PCA-ready matrix X from Task 4 (no NaNs, standardized; shape: (n_samples, 14)).
    - Fit PCA with n_components=3.
    - Return TWO arrays:
        1) explained_variance_ratio_ -> shape (3,)
        2) components_               -> shape (3, 14)
    """

    # TODO 1 — init PCA

    # TODO 2 — fit

    # TODO 3 — grab arrays
    
    # TODO 4 — return

In [206]:
pca = PCA(n_components=3)

In [207]:
pca.fit(X_scaled)

,n_components,3
,copy,True
,whiten,False
,svd_solver,'auto'
,tol,0.0
,iterated_power,'auto'
,n_oversamples,10
,power_iteration_normalizer,'auto'
,random_state,None


In [212]:
print(pca.explained_variance_ratio_.shape)

(3,)


In [209]:
print(dir(pca))

['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__sklearn_clone__', '__sklearn_tags__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_build_request_for_signature', '_doc_link_module', '_doc_link_template', '_doc_link_url_param_generator', '_fit', '_fit_full', '_fit_svd_solver', '_fit_truncated', '_get_default_requests', '_get_doc_link', '_get_metadata_request', '_get_param_names', '_get_params_html', '_html_repr', '_n_features_out', '_parameter_constraints', '_repr_html_', '_repr_html_inner', '_repr_mimebundle_', '_sklearn_auto_wrap_output_keys', '_transform', '_validate_params', 'components_', 'copy', 'expl

In [211]:
print(pca.components_.shape)

(3, 14)


In [ ]:
# Reload again to reset any changes
importlib.reload(hw)

In [213]:
ratio, comps = pca.explained_variance_ratio_, pca.components_
print('variance ratios:', ratio, 'sum:', ratio.sum())
features = hw.PCA_FEATURES
for i, row in enumerate(comps):
    j = np.argmax(np.abs(row))
    print(f'PC{i+1} top feature: {features[j]} (loading={row[j]:.3f})')

variance ratios: [0.17258335 0.10995725 0.09686639] sum: 0.37940697823552316
PC1 top feature: total_guests (loading=0.595)
PC2 top feature: stays_in_week_nights (loading=0.571)
PC3 top feature: lead_time (loading=0.366)


In [225]:
# np.abs(comps[0])
for index,i in enumerate(comps):
    # print(index)
    required_index = np.argmax(np.abs(i))
    print("PCA:",index+1,"top feature",features[required_index],"contribution ",round(float(i[required_index]),3))
    

PCA: 1 top feature total_guests contribution  0.595
PCA: 2 top feature stays_in_week_nights contribution  0.571
PCA: 3 top feature lead_time contribution  0.366


In [226]:
# ⚠️ Do not change this part!  
# These assertions mirror the VPL auto-grading checks.  
# Use them locally to confirm that your function works properly.

assert ratio.shape == (3,) and comps.shape == (3, 14)
print("All local checks passed!")

All local checks passed!



## ✨ Optional Practice (Not Graded)
---

This part is **not graded**, but it is highly recommended as extra practice.  

Congratulations! You’ve completed your assignment as a **data science consultant** for the hotels. You have:
- Cleaned and structured messy booking data,  
- Computed KPIs to get a clear picture of cancellations,  
- Explored patterns across features,  
- Applied PCA to simplify and highlight the most important drivers.  

Now comes the most important part: **communication**.  
The hotel managers don’t want code or plots! they want a clear story. As a consultant, your analysis is only as valuable as the clarity of your recommendations.  
Take a few minutes to **summarize your findings in plain language** as if you were writing a short note to the hotel managers.  

Your note should answer questions like:  
- What are the **main reasons for cancellations**?  
- Which factors appear less important?  
- What practical advice can managers take away?  

Keep it short: 5–7 sentences is enough. Managers don’t have time to read a long report with many pages! 😄 

Think of this as practicing the skill of **turning data into decisions**, which is just as important as the analysis itself.  

This closes the loop: from raw data → structured workflow → actionable business insight.  
